### 1. Coletando dados

Na prática, a ingestão raramente começa com objetos prontos. Ela começa com arquivos, páginas, PDFs, HTML ou texto extraído de alguma fonte. Aqui vamos simular três livros como texto puro.

In [1]:
book1 = """
TÍTULO: Arriscando a Própria Pele
AUTOR: Nassim Taleb
ANO: 2018
PÁGINA 12
Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.
PÁGINA 41
Sistemas robustos precisam considerar assimetria, incentivos e exposição real ao erro.
"""

book2 = """
TÍTULO: Antifrágil
AUTOR: Nassim Taleb
ANO: 2012
PÁGINA 8
Alguns sistemas melhoram quando passam por volatilidade, pressão e incerteza.
PÁGINA 77
Decisão sob incerteza exige avaliar risco antes de aceitar perdas irreversíveis.
"""

book3 = """
TÍTULO: Designing Data-Intensive Applications
AUTOR: Martin Kleppmann
ANO: 2017
PÁGINA 23
Sistemas distribuídos precisam lidar com falhas parciais, latência e replicação.
PÁGINA 67
Bons modelos de dados tornam consultas mais previsíveis e fáceis de evoluir.
"""

raw_books = [book1, book2, book3]

print(f"Quantidade de livros: {len(raw_books)}")

print("\nLivro 1 [PREVIEW]:\n")
print(raw_books[0].strip()[:155])


Quantidade de livros: 3

Livro 1 [PREVIEW]:

TÍTULO: Arriscando a Própria Pele
AUTOR: Nassim Taleb
ANO: 2018
PÁGINA 12
Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolha


### 2. Extraindo estrutura

O parser abaixo é simples de propósito. Ele transforma texto bruto em um objeto com título, autor, ano e páginas. Em um sistema real, essa etapa poderia envolver OCR, parsers de HTML, extração de PDF e validações de qualidade.

In [2]:
import json
import re

def parse_raw_book(raw_text):
    title = re.search(r"TÍTULO:\s*(.+)", raw_text).group(1).strip()
    author = re.search(r"AUTOR:\s*(.+)", raw_text).group(1).strip()
    year = int(re.search(r"ANO:\s*(\d{4})", raw_text).group(1))
    page_pattern = r"PÁGINA\s+(\d+)\s+(.+?)(?=\n\s*PÁGINA\s+\d+|\Z)"
    page_matches = re.findall(page_pattern, raw_text, flags=re.S)
    pages = [
        {"page": int(page), "text": " ".join(text.split())}
        for page, text in page_matches
    ]
    return {
        "title": title,
        "author": author,
        "year": year,
        "pages": pages,
    }

books = [parse_raw_book(raw_book) for raw_book in raw_books]
print(json.dumps(books[0], ensure_ascii=False, indent=2))

{
  "title": "Arriscando a Própria Pele",
  "author": "Nassim Taleb",
  "year": 2018,
  "pages": [
    {
      "page": 12,
      "text": "Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas."
    },
    {
      "page": 41,
      "text": "Sistemas robustos precisam considerar assimetria, incentivos e exposição real ao erro."
    }
  ]
}


### 3. Criando unidades recuperáveis

Depois do parse, cada página vira um registro buscável. Como temos 3 livros com 2 páginas cada, o corpus termina com 6 registros. O campo `search_text` combina título, autor e texto da página; os metadados ficam separados para filtro e retorno. Em seguida, o corpus aplica a normalização definida por `preprocess()` para gerar o texto efetivamente usado na busca.

In [3]:
def preprocess(text):
    tokens = re.findall(r"\w+", text.lower())
    return [token for token in tokens if len(token) > 1]

def parse_book(book):
    registros = []
    for page in book["pages"]:
        registros.append({
            "id": f"{book['title']}:p{page['page']}",
            "text": page["text"],
            "search_text": f"{book['title']} {book['author']} {page['text']}",
            "metadata": {
                "title": book["title"],
                "author": book["author"],
                "year": book["year"],
                "page": page["page"],
            },
        })
    return registros

def parse_books(books):
    registros = []
    for book in books:
        registros.extend(parse_book(book))
    return registros

registros = parse_books(books)
print(f"Registros buscáveis: {len(registros)}")
print("Livro 1 formatado:")
print(json.dumps(registros[0], ensure_ascii=False, indent=2))

corpus = [" ".join(preprocess(registro["search_text"])) for registro in registros]

print(f"Corpus item 0:\n > {corpus[0]}")
print(f"Corpus item 1:\n > {corpus[1]}")
print(f"Corpus item 2:\n > {corpus[2]}")


Registros buscáveis: 6
Livro 1 formatado:
{
  "id": "Arriscando a Própria Pele:p12",
  "text": "Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.",
  "search_text": "Arriscando a Própria Pele Nassim Taleb Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.",
  "metadata": {
    "title": "Arriscando a Própria Pele",
    "author": "Nassim Taleb",
    "year": 2018,
    "page": 12
  }
}
Corpus item 0:
 > arriscando própria pele nassim taleb risco decisão mudam quando pessoa sofre as consequências das próprias escolhas
Corpus item 1:
 > arriscando própria pele nassim taleb sistemas robustos precisam considerar assimetria incentivos exposição real ao erro
Corpus item 2:
 > antifrágil nassim taleb alguns sistemas melhoram quando passam por volatilidade pressão incerteza


### 4. Indexando o corpus

Para manter o exemplo concreto, vamos usar TF-IDF como uma função de relevância lexical. Por enquanto, não precisamos entender o cálculo em detalhe. Basta pensar nele como uma forma de medir quão relevantes são os termos da pergunta dentro do corpus: termos presentes na query, no documento e na coleção ajudam a produzir um score de busca. O próximo artigo abre essa ideia com calma.

Neste ponto, o mais importante é observar que a ingestão transformou registros em uma estrutura consultável. A dimensão da matriz mostra quantos registros existem e quantos termos únicos foram usados para representá-los.

In [4]:
import math
from collections import Counter

def build_tfidf_matrix(corpus):
    tokenized_documents = [document.split() for document in corpus]
    vocabulary = sorted({token for document in tokenized_documents for token in document})
    document_count = len(tokenized_documents)
    document_frequency = {
        term: sum(1 for document in tokenized_documents if term in document)
        for term in vocabulary
    }
    idf = {
        term: math.log((1 + document_count) / (1 + document_frequency[term])) + 1
        for term in vocabulary
    }
    matrix = []
    for tokens in tokenized_documents:
        counts = Counter(tokens)
        total_terms = sum(counts.values()) or 1
        matrix.append([(counts[term] / total_terms) * idf[term] for term in vocabulary])
    return vocabulary, idf, matrix

vocabulary, idf, tfidf_matrix = build_tfidf_matrix(corpus)

print(f"shape da matriz: ({len(tfidf_matrix)}, {len(vocabulary)})")
print(f"{len(tfidf_matrix)} linhas: cada página dos 3 livros virou um registro buscável")
print(f"{len(vocabulary)} colunas: termos únicos do corpus")


shape da matriz: (6, 64)
6 linhas: cada página dos 3 livros virou um registro buscável
64 colunas: termos únicos do corpus


### 5. Recuperando candidatos

Quando a query chega, o sistema transforma a busca para a mesma representação, compara com a matriz e ordena candidatos. Neste exemplo, a busca também recebe `filters={"author": "Nassim Taleb"}`. Isso limita o ranking final aos registros desse autor, mesmo que o corpus tenha livros de outros autores.

Novamente, não se preocupe com as fórmulas matemáticas por enquanto. Concentre-se na ideia: encontrar páginas possivelmente relevantes a partir dos termos presentes na pergunta e no conteúdo dos livros.

In [5]:
def vectorize_query(query, vocabulary, idf):
    tokens = preprocess(query)
    counts = Counter(tokens)
    total_terms = sum(counts.values()) or 1
    return [(counts[term] / total_terms) * idf.get(term, 0) for term in vocabulary]

def cosine_similarity(vector_a, vector_b):
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    norm_a = math.sqrt(sum(value * value for value in vector_a))
    norm_b = math.sqrt(sum(value * value for value in vector_b))
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot_product / (norm_a * norm_b)

def matches_filters(registro, filters):
    if not filters:
        return True
    return all(registro["metadata"].get(key) == value for key, value in filters.items())

def search_tfidf(query, top_k=3, filters=None):
    query_vector = vectorize_query(query, vocabulary, idf)
    scores = [cosine_similarity(document_vector, query_vector) for document_vector in tfidf_matrix]
    candidates = [
        (registro, scores[index])
        for index, registro in enumerate(registros)
        if matches_filters(registro, filters)
    ]
    return sorted(candidates, key=lambda item: item[1], reverse=True)[:top_k]

results = search_tfidf("Arriscando Taleb risco decisão", filters={"author": "Nassim Taleb"})

for registro, score in results:
    metadata = registro["metadata"]
    print(f"{metadata['title']} | p. {metadata['page']} | score={score:.2f}")
    print(registro["text"])
    print()


Arriscando a Própria Pele | p. 12 | score=0.43
Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.

Antifrágil | p. 77 | score=0.33
Decisão sob incerteza exige avaliar risco antes de aceitar perdas irreversíveis.

Arriscando a Própria Pele | p. 41 | score=0.19
Sistemas robustos precisam considerar assimetria, incentivos e exposição real ao erro.

